# **Build the fist MIL pipeline : 1D CNN + Classifier**

## **Defining the training on the GPU**

In [1]:
from pathlib import Path
CHECKPOINT_DIR = Path("checkpoints_mil_128_classifier_3s")
import mlflow
from Tracking.mlflow_config import set_project_experiment

c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
experiment_name = "MIL128_Model_3s_"

In [3]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print("Dossier des checkpoints :", CHECKPOINT_DIR.resolve())

Dossier des checkpoints : C:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\Tests\checkpoints_mil_128_classifier_3s


In [4]:
import torch.nn as nn
import numpy as np
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt 

In [5]:
import os
# Enable precise CUDA kernel error reporting
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"
# Import PyTorch AFTER setting the environment variables
import torch

In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Device:", device)
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable; refusing to train on CPU.")
print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


## **Building the dataset**

In [7]:
from Data_pre_processing.data_cleaning import data_cleaning, split_train_test
import torch
from torch.utils.data import Dataset
from Data_pre_processing.Signal_treatement import split_signals_and_build

### **Getting the data**

La fonction upload_data retourn deux tuples : 
- Le premier tuple (training_x, training_y)
- Le deuxieme tuple (testing_x, testing_y)

donc pour acceder il suffit d'utiliser les indices 0 et 1, exemple : 

- training[0] donne le training_x, training[1] donne le training_y

In [8]:
from Data_pre_processing.Data_pipeline import upload_data

In [9]:
training, testing = upload_data(segment_duration=3) 

Normal: 522
Abnormal: 426


### **Building the dataset using PyTorch for training**

In [10]:
from Data_pre_processing.Build_dataset import PCGDataset

In [11]:
Training_data = PCGDataset(bags = training[0], labels = training[1])
Testing_data = PCGDataset(bags = testing[0], labels = testing[1])

## **Training**

- Pour l'entrainement on a le problème d'umblanced data, donc pour valider le modele, on vas utiliser le stratified cross validation.
- Le nombre de folds qu'on vas utiliser maintenant c'est 5.
- 1 pour abnormal, 0 pour normal.
- In this case we are going to priviluaze our folds to see the data distribution

### **Stratified cross validation**

In [12]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

### **Training phase**

In [13]:
from Data_pre_processing.Build_dataset import build_data_loaders
from Models.MIL1D_128 import Mil_classifier_128
from Training.Train_model import TrainModel

#### **Training and validation**

Ici je défini les Hyper parametres :

In [14]:
NUM_EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
fold_results = []
MOMENTUM = 1e-4

Phase d'entrainement :

In [19]:
folds = []
pos_weight = torch.tensor([0.5], device=device)
for train_index, test_index in skf.split(
    np.zeros(len(training[1])),
    training[1],
):
    folds.append((train_index, test_index))

articraft_dir = set_project_experiment("PCG_Heart_MIL_128_3s_3Folds")
articraft_dir = articraft_dir / "Articrafts"
articraft_dir.mkdir(parents=True, exist_ok=True)

for fold, (train_idx, valid_idx) in enumerate(folds) : 
    print(f"training on the fold {fold}")
    print(f"*"*20, "INFORMATION ABOUT THE TRAINING", "*"*20)
    experiment_name_fold = f"{experiment_name}3Folds_fold{fold}"
    #MLflow new experiment
    numbers_of_zero = 0
    numbers_of_one = 0
    for index in train_idx :
      if training[1][index] == 1 :
        numbers_of_one+=1
      else : numbers_of_zero+=1
      
    print(f"      numbers of abnormal : {numbers_of_one}, numbers of normal : {numbers_of_zero}")

    torch.manual_seed(100 + fold)

    model = Mil_classifier_128().to(device=device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    checkpoint_path = CHECKPOINT_DIR / f"fold_{fold}.pt"

    start_epoch = 0

    if checkpoint_path.exists():
        checkpoint = torch.load(
            checkpoint_path,
            map_location=device,
            weights_only=True
        )

        if checkpoint["fold"] != fold:
            raise ValueError("Le checkpoint ne correspond pas au fold.")

        model.load_state_dict(
            checkpoint["model_state_dict"]
        )

        optimizer.load_state_dict(
            checkpoint["optimizer_state_dict"]
        )

        start_epoch = checkpoint["epoch"]

        print(
            f"Reprise du fold {fold} : "
            f"dernière époque terminée = {start_epoch}"
        )
    else:
        print(f"New training for the fold {fold}")

    train_loader, valid_loader = build_data_loaders(dataset=Training_data, train_index=train_idx, test_index=valid_idx)
    
    with mlflow.start_run(run_name=experiment_name_fold):
        
        mlflow.log_params({
            "optimizer" : "AdamW",
            "learning_rate" : LEARNING_RATE,
            "Momentum" : MOMENTUM,
            "Epochs" : NUM_EPOCHS,
            "weight_decay" : WEIGHT_DECAY,
            "pos_weight" : pos_weight,
            "early_stopping" : 20,
        })

        Training_model = TrainModel(
           model=model,
           criterion=criterion,
           optimizer=optimizer,
           train_loader=train_loader,
           device=device,
           valid_loader=valid_loader,
           start_epoch=start_epoch,
           number_epochs=NUM_EPOCHS,
           checkpoint_path=checkpoint_path,
           patience=10,
           min_delta = 0.001,
        )

        loss_res_train, loss_res_validation = Training_model.train_model(fold=fold)

        plt.plot(loss_res_train, label="Train loss")
        plt.plot(loss_res_validation, label="Validation loss")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.title(f"Training and Validation Loss for the fold {fold}")
        plt.legend()
        
        plt.savefig(f"{articraft_dir}/loss_curve.png")
        plt.close()

        mlflow.log_artifact(f"{articraft_dir}/loss_curve.png")
        
     

training on the fold 0
******************** INFORMATION ABOUT THE TRAINING ********************
      numbers of abnormal : 340, numbers of normal : 418
New training for the fold 0


c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Fold : 0 | Train loss : 0.4203 | Train accuracy : 0.6332 | Validation loss : 0.4810 | Validation accuracy : 0.5474 | epoch : 1/30
checkpoint saved .. !


c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Fold : 0 | Train loss : 0.4122 | Train accuracy : 0.6359 | Validation loss : 0.4157 | Validation accuracy : 0.5474 | epoch : 2/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.3997 | Train accuracy : 0.6900 | Validation loss : 0.4187 | Validation accuracy : 0.6368 | epoch : 3/30
Fold : 0 | Train loss : 0.3978 | Train accuracy : 0.7124 | Validation loss : 0.4136 | Validation accuracy : 0.6421 | epoch : 4/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.3984 | Train accuracy : 0.7137 | Validation loss : 0.4075 | Validation accuracy : 0.6526 | epoch : 5/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.3955 | Train accuracy : 0.7243 | Validation loss : 0.4005 | Validation accuracy : 0.6684 | epoch : 6/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.3981 | Train accuracy : 0.7137 | Validation loss : 0.3940 | Validation accuracy : 0.6684 | epoch : 7/30
checkpoint saved .. !
Fold : 0 | Train loss : 0.3937 | Train accuracy : 0.7203 | Validation loss : 0.3883 | Validation accuracy : 0.

c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


... (14 outputs left out)

Fold : 1 | Train loss : 0.3918 | Train accuracy : 0.7045 | Validation loss : 0.4566 | Validation accuracy : 0.5526 | epoch : 8/30
Fold : 1 | Train loss : 0.3892 | Train accuracy : 0.7084 | Validation loss : 0.4574 | Validation accuracy : 0.5684 | epoch : 9/30


c:\Users\aliai\Desktop\M2 I2A\Optimisation des modeles - Ouzar\Projet\venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Fold : 1 | Train loss : 0.3869 | Train accuracy : 0.7098 | Validation loss : 0.4548 | Validation accuracy : 0.5526 | epoch : 10/30
Fold : 1 | Train loss : 0.3858 | Train accuracy : 0.7098 | Validation loss : 0.4546 | Validation accuracy : 0.5684 | epoch : 11/30
Fold : 1 | Train loss : 0.3841 | Train accuracy : 0.7098 | Validation loss : 0.4530 | Validation accuracy : 0.5737 | epoch : 12/30
checkpoint saved .. !
Fold : 1 | Train loss : 0.3840 | Train accuracy : 0.7098 | Validation loss : 0.4514 | Validation accuracy : 0.5737 | epoch : 13/30
checkpoint saved .. !
Fold : 1 | Train loss : 0.3824 | Train accuracy : 0.7150 | Validation loss : 0.4502 | Validation accuracy : 0.5737 | epoch : 14/30
checkpoint saved .. !
Fold : 1 | Train loss : 0.3824 | Train accuracy : 0.7150 | Validation loss : 0.4511 | Validation accuracy : 0.5737 | epoch : 15/30
Fold : 1 | Train loss : 0.3797 | Train accuracy : 0.7150 | Validation loss : 0.4471 | Validation accuracy : 0.5789 | epoch : 16/30
checkpoint saved 

KeyboardInterrupt: 